# Exact verification: diagonal saturation model
Reproduces the active-space geometry, sharp perturbation bound, regularization certificate, and local spectral-rate comparison used in the manuscript.

In [ ]:
import numpy as np

Delta, W = 1.0, 4.0
G = np.diag([0.25, 4.0])
Hact = np.diag([0.5, 32.0])
gminus, gplus = np.linalg.eigvalsh(G)[[0,-1]]

def invsqrt(A):
    w,V=np.linalg.eigh(A)
    return V@np.diag(1/np.sqrt(w))@V.T

S0=invsqrt(G)@Hact@invsqrt(G)
print('spec(S0)=',np.linalg.eigvalsh(S0))
print('kappa(S0)=',np.linalg.cond(S0),'  W/Delta=',W/Delta)
assert np.allclose(np.linalg.eigvalsh(S0),[2,8])

eps=0.1
lam=eps
Xi=np.diag([0.1,-0.1])
M=G+Xi+lam*np.eye(2)
Sl=invsqrt(M)@Hact@invsqrt(M)
K=(W/Delta)*(1+2*eps/gminus)
print('spec(S_lambda)=',np.linalg.eigvalsh(Sl))
print('kappa(S_lambda)=',np.linalg.cond(Sl),'  K=',K)
assert np.isclose(np.linalg.cond(Sl),7.2)
assert np.isclose(K,7.2)

kappaH=np.linalg.cond(Hact)
epscrit=gminus/2*((Delta/W)*kappaH-1)
rho_geom=(np.linalg.cond(Sl)-1)/(np.linalg.cond(Sl)+1)
rho_gd=(kappaH-1)/(kappaH+1)
print('kappa(H)=',kappaH,' eps_crit=',epscrit)
print('rho_geom=',rho_geom,' rho_GD=',rho_gd)
assert eps<epscrit and K<kappaH and rho_geom<rho_gd

# Symmetric perturbation stress test
rng=np.random.default_rng(20260910)
max_kappa=0.0
for _ in range(500):
    A=rng.normal(size=(2,2)); A=(A+A.T)/2
    A=A/np.linalg.norm(A,2)*eps
    M=G+A+lam*np.eye(2)
    S=invsqrt(M)@Hact@invsqrt(M)
    max_kappa=max(max_kappa,np.linalg.cond(S))
print('max stress-test kappa=',max_kappa,' <= K=',K)
assert max_kappa<=K+1e-10
